# Sustainable Race Calendar Optimization for Formula One

#### CS 524: Introduction to Optimization Fall 2025

#### Authors: 
  - *Gokulnath Sourirajan*
  - *Thilak Raj Murugan*


## 1. Introduction

Formula One (F1) has committed to achieving net-zero carbon emissions by 2030, yet a major share of its footprint still comes from the extensive global travel required each season. The current race calendar forces teams to repeatedly fly and ship equipment across continents, creating unnecessary logistical emissions. This project proposes a data-driven approach to redesigning the F1 calendar with sustainability in mind, using geospatial, climate, and cultural datasets to ensure any optimized schedule remains both practical and operationally realistic.


### 1.1 Problem Statement

The goal is to formulate the F1 race schedule as a routing optimization problem that minimizes total logistics-related carbon emissions. Each race venue is defined by its location, feasible weather window, and travel cost determined by distance and transport mode. 

The optimized calendar must respect real-world constraints such as fixed opening and closing races, atmost 3 consecutive events, and a mid-season break—while remaining operationally feasible. By solving this constrained optimization problem, the project aims to produce a race calendar that has the minimum CO2 emissions.


## 2. Approach

The model is built using sets of circuits and weekends, with binary decision variables that determine both the race order and the specific weekend assigned to each event. Additional variables capture positional ordering (via Miller–Tucker–Zemlin constraints), triple-header occurrences, long gaps between races, and the boundary around the summer break.

The constraints enforce the Traveling Salesman Problem (TSP) structure, wherein each circuit has exactly one predecessor and one successor, and each race is scheduled exactly once. We also ensure at most one race per weekend and eliminate subtours. Feasibility rules prevent races from being placed on weekends restricted by climate or cultural conditions. Temporal constraints link the race sequence with actual weekend assignments.

The model also embeds operational requirements: exactly 14 races before the summer break, races immediately before and after the break, and no races during the break period. Additional constraints control the number of triple-headers and limit excessively long gaps between races. Break-arc detection variables identify the transition cut by the break to correctly adjust emissions in the objective function. Overall, this formulation produces a season-long schedule that satisfies all logistical, climatic, and operational constraints while minimizing total freight-related emissions.

### 2.1 Modeling the summer break

We modelled the summer break by enforcing the below constraints:

- Exactly 14 races must be scheduled before the F1 summer break.
- The summer break spans (2026-08-09 to 2026-08-30) and no races can occur during this interval.
- A race must be scheduled on the weekends immediately before and after the summer break.

The summer break arc is the edge in the race graph that connects the last race before the summer break to the first race after the break. In reality, this arc does not exist, as teams travel back to their respective headquarters during the break. We use constraints on z[i, j] to identify this break arc. To model this correctly in the objective function, we subtract the emissions associated with the break arc, and instead add the emissions from pre-break race to HQ and the emission from HQ to post-break race.

### 2.2 Objective Function.

The total emission is calculated by taking the sum of the following items:

- **Start-of-season travel**: Emissions for all teams traveling from their respective HQ to the first race location.
- **Regular season travel**: Emissions for consecutive race-to-race travel, multiplied by 10 as there are 10 teams.
- **End-of-season travel**: Emissions for teams traveling from the final race back to their respective HQ.
- **Summer break emissions**:
    - Subtract emissions of the break arc.
    - Add emissions for pre-break race -> HQ and HQ -> post-break race travel.

### 2.2 Assumptions

#### 2.2.1 Representation of Race Locations

Each race location is represented by the geographical coordinates of its host city, serving as a proxy for the nearest international airport. This captures the primary logistics hubs used by Formula 1 teams without explicitly modeling short local transfers.

#### 2.2.2 Emission Factors and Freight Load

- Each team transports a standardized freight load of 50 tonnes.
- Emissions are calculated using tonne-kilometer factors:
    - Air: 0.13516 kg CO₂e/tonne-km
	- Road: 0.0165 kg CO₂e/tonne-km

#### 2.2.3 Modes of Transport

Only two logistics modes are modeled: air (for intercontinental/non-European transfers) and road (for intra-European transfers).

Furthermore, we made the following assumptions in our model:

- The Haversine distance is used to approximate flight emissions, assuming symmetric travel paths.
- A constant emission factor (kgCO₂e per km) is applied, with air-freight considered the dominant contributor.
- The opening and closing races are fixed as Australia and Abu Dhabi, respectively.
- Weekends with climate or cultural infeasibility are strictly disallowed.
- Triple-header limits and maximum gap constraints must be respected to account for real-world scheduling fatigue.
- The analysis includes only CO₂ emissions from freight transport between race locations and doesn't include other green house gases.

### 2.3 Data Collection & Preprocessing

#### 2.3.1 Circuit & Team Locations

- Circuit coordinates are stored in `circuits.csv`.
- Team headquarters are stored in `hq.csv`.
- Haversine distances between circuits, and between a circuit and HQ, are computed and stored in `distances.csv` .
- Climate data are stored in `climate.csv`
- Festival data are stored in `festival.csv`. These dates are marked infeasible in the feasibility matrix.

#### 2.3.2 Weather Feasibility

- For each circuit, five years of historical daily weather data were collected using the
Visual Crossing API.
- For each Sunday of 2026, the average temperature and precipitation across the past 5
years were computed.
- A weekend is marked feasible (1) if:
    - 10°C ≤ temperature ≤ 35°C
    - precipitation < threshold


## 3.1 GAMSPy Model

In [14]:
import sys
import numpy as np
import pandas as pd
import gamspy as gp

gp.set_options({'USE_PY_VAR_NAME': 'yes'})

# Emission factors (kg CO2e per tonne-km) – Climatiq (https://www.climatiq.io/)
EMISSION_FACTORS_TONNE_KM = {
    "air": 0.13516,
    "land": 0.0165
}

# Average freight load per team (tonnes)
AVERAGE_TEAM_FREIGHT_TONNES = 50

# Weather feasibility thresholds
MIN_TEMP_F = 50      # Minimum ambient temperature for race viability
MAX_TEMP_F = 95      # Maximum ambient temperature for race viability
MAX_PRECIP_IN = 1.57  # Maximum weekly precipitation for race feasibility

In [15]:
# LOAD DATA
teams_df = pd.read_csv("./hq.csv")
circuits_df = pd.read_csv("./circuits.csv")
distances_df = pd.read_csv("./distances.csv")
climate_df = pd.read_csv("./climate.csv")
festivals_df = pd.read_csv("./festivals.csv")

# Declare variables for printing baseline model solution
x_baseline = None
u_baseline = None
y_baseline = None


In [16]:
def define_and_run_model(FIRST="AUS", LAST="ABD", is_visualize=False, time_limit=300):
    m = gp.Container()

    rev_df = distances_df.rename(columns={"from": "to", "to": "from"})
    distances_sym_df = (
        pd.concat([distances_df, rev_df], ignore_index=True)
          .drop_duplicates(subset=["from", "to"])
    )
    
    # CONSTANTS
    BREAK_START_DATE = 24 # 2026-08-09 (in climate_df)
    BREAK_END_DATE = 26   # 2026-08-30 (in climate_df)

    
    # Identify last weekend before break and first after break
    t_pre  = str(BREAK_START_DATE - 1)
    t_post = str(BREAK_END_DATE   + 1)
    
    MAX_TRIPLE_HEADERS = 3
    
    # PREPROCESSING
    
    teams = teams_df["team_id"].tolist()[:-1]
    circuits = circuits_df["circuit_id"].tolist()
    
    weekends = climate_df['week_num'].astype(int).tolist()
    summer_break = climate_df.loc[
        climate_df['week_num'].between(BREAK_START_DATE, BREAK_END_DATE), 
        'week_num'].astype(int).tolist()
    
    
    race_emissions = distances_sym_df[distances_sym_df['from'].isin(circuits) & 
                                      distances_sym_df['to'].isin(circuits)][["from", "to", "emissions_kgCO2e"]]
    hq_emissions = distances_sym_df[distances_sym_df['from'].isin(teams)][["from", "to", "emissions_kgCO2e"]]
    
    n_teams = len(teams)
    n_races = len(circuits)
    n_weekends = len(weekends)
    
    feasible_dates = np.zeros((n_weekends, n_races))
    feasible_dates_df = pd.DataFrame(feasible_dates, columns=circuits, index=climate_df['week_num'].astype(int).tolist())
    
    climate_df.index = climate_df['week_num'].astype(int).tolist()
    
    for circuit in circuits:
        temp_col = f"{circuit}_avg_temp"
        precip_col = f"{circuit}_avg_precip"
    
        ok_temp = climate_df[temp_col].between(MIN_TEMP_F, MAX_TEMP_F)
        ok_precip = climate_df[precip_col] < MAX_PRECIP_IN
    
        feasible_dates_df.loc[ok_temp & ok_precip, circuit] = 1
    
    # Festival filter
    for idx, row in festivals_df.iterrows():
        feasible_dates_df.iloc[row['week_num'] -1 , feasible_dates_df.columns.get_loc(row['circuit_id'])] = 0
    
    # SET
    Weekend = gp.Set(m, records=weekends)
    SummerBreak = gp.Set(m, domain=[Weekend], records=summer_break)
    Team = gp.Set(m, records=teams)
    Circuit = gp.Set(m, records=circuits)
    i = gp.Alias(m, alias_with=Circuit)
    j = gp.Alias(m, alias_with=Circuit)
    t = gp.Alias(m, alias_with=Weekend)
    
    
    
    # PARAMETERS
    r_emissions = gp.Parameter(m, domain=[Circuit, Circuit], records=race_emissions,
                             description="emissions between races in kgCO2e")
    hq_emissions = gp.Parameter(m, domain=[Team, Circuit], records=hq_emissions,
                             description="emissions from HQ to race in kgCO2e")
    
    feasible_dates = gp.Parameter(m, domain=[Weekend, Circuit], records=feasible_dates_df.stack().reset_index().values.tolist(),
                                description="feasibility of holding race on weekend (1 if feasible, 0 otherwise)")
    feasible_dates[SummerBreak, Circuit] = 0 # No races during the break
        
    
    first_set = gp.Parameter(m, records=14,
                             description='Number of races before the summer break')
    break_start_date = gp.Parameter(m, records=np.array(BREAK_START_DATE),
                                    description='Weekend where break begins')
    break_end_date = gp.Parameter(m, records=np.array(BREAK_END_DATE),
                                    description='Weekend where break ends')
    
    # VARIABLES
    x = gp.Variable(m,type='binary',domain=[Circuit,Circuit],
                    description="1 if race in i is scheduled immediately before j, 0 otherwise")
    y = gp.Variable(m, type='binary', domain=[Circuit,Weekend], 
                    description='1 if race in Circuit is scheduled on Weekend, 0 otherwise')
    z = gp.Variable(m, type="binary", domain=[i, j],
                    description="1 if race i is on pre-break weekend and race j on post-break weekend")
    u = gp.Variable(m,type='positive',domain=[Circuit], 
                    description="Position of race in the calendar sequence")
    gap = gp.Variable(m, type='binary', domain=[t],
                      description="1 if no race at week t")
    triple_header = gp.Variable(m, type='binary', domain=[t],
                                description="1 if week t starts a triple header")
    
    
    x.fx[i,i] = 0
    
    u.lo[Circuit] = 2                   # all races but first must be at least position 2
    u.up[Circuit] = gp.Card(Circuit)    # all races must be at most position number of races
    
    u.fx[FIRST] = 1                # First race = AUS
    u.fx[LAST] = gp.Card(Circuit)  # Last race = ABD (position = number of races)
    
    
    
    # EQUATIONS
    assign1 = gp.Equation(m, domain=[j],
                          description='Each circuit has exactly one predecessor')
    assign1[j] = gp.Sum(i, x[i,j]) == 1
    
    assign2 = gp.Equation(m, domain=[i],
                          description='Each circuit has exactly one successor')
    assign2[i] = gp.Sum(j, x[i,j]) == 1
    
    assign3 = gp.Equation(m, domain=[i],
                          description='Each race is held exactly once')
    assign3[i] = gp.Sum(t, y[i,t]) == 1
    
    assign4 = gp.Equation(m, domain=[t],
                          description='Each weekend has at most one race')
    assign4[t] = gp.Sum(i, y[i,t]) <= 1
    
    # Get actual MTZ ordinal positions
    FIRST_ord = circuits.index(FIRST) + 1
    LAST_ord = circuits.index(LAST) + 1
    
    # Then use ordinals instead of strings
    mtz = gp.Equation(m, domain=[i,j])
    mtz[i,j].where[(i.ord != FIRST_ord) & (j.ord != FIRST_ord) & (i.ord != LAST_ord)] = (
        u[i] - u[j] + 1 <= (gp.Card(Circuit) - 1) * (1 - x[i,j])
    )
    
    # Time Constraints
    feasibiliy = gp.Equation(m, domain=[i,t],
                             description='Race can be held only if the weekend is feasible')
    feasibiliy[i,t] = y[i,t] <= feasible_dates[t,i]
    
    
    time_link = gp.Equation(m, domain=[i,j])
    time_link[i,j].where[(i.ord != LAST_ord) | (j.ord != FIRST_ord)] = (
        gp.Sum(t, t.ord * y[j,t]) >= gp.Sum(t, t.ord * y[i,t]) + x[i,j] - (1 - x[i,j]) * gp.Card(Weekend)
    )
    
    # Triple Header Limits
    triple_header_upper = gp.Equation(m, domain=[t],
                                      description='Direction 1: If triple_header=1, then must have 3 consecutive races')
    triple_header_upper[t].where[t.ord <= gp.Card(Weekend) - 2] = (
        triple_header[t] * 3 <= gp.Sum(i, y[i, t]) + gp.Sum(i, y[i, t.lead(1)]) + gp.Sum(i, y[i, t.lead(2)])
    )
    
    triple_header_lower = gp.Equation(m, domain=[t],
                                      description='Direction 2: If there are 3 consecutive races, triple_header MUST be 1')
    triple_header_lower[t].where[t.ord <= gp.Card(Weekend) - 2] = (
        gp.Sum(i, y[i, t]) + gp.Sum(i, y[i, t.lead(1)]) + gp.Sum(i, y[i, t.lead(2)]) <= 2 + triple_header[t]
    )
    
    triple_header_limit = gp.Equation(m,
                                      description='Limit on total triple headers')
    triple_header_limit[...] = gp.Sum(t, triple_header[t]) <= MAX_TRIPLE_HEADERS
    
    
    
    # Mid Season Break Condition
    break_partition = gp.Equation(m,
                                  description='Exactly 14 races before summer break')
    break_partition[...] = gp.Sum([i, t], y[i,t].where[t.ord < BREAK_START_DATE]) == first_set
    
    pre_break_race = gp.Equation(m,
                                 description='Exactly one race on last weekend before break')
    pre_break_race[...] = gp.Sum(i, y[i, str(BREAK_START_DATE - 1)]) == 1
    
    post_break_race = gp.Equation(m,
                                  description='Exactly one race on first weekend after break')
    post_break_race[...] = gp.Sum(i, y[i, str(BREAK_END_DATE + 1)]) == 1
    
    
    # Nice Gaps between races
    gap_def = gp.Equation(m, domain=[t])
    gap_def[t] = gap[t] == 1 - gp.Sum(i, y[i, t])
    
    max_consecutive = gp.Equation(m, domain=[t])
    max_consecutive[t].where[t.ord <= gp.Card(Weekend) - 3] = (
        gap[t] + gap[t.lead(1)] + gap[t.lead(2)] + gap[t.lead(3)] >= 1
    )
    
    
    z_lin1 = gp.Equation(m, domain=[i, j])
    z_lin1[i, j] = z[i, j] <= x[i, j]
    
    z_lin2 = gp.Equation(m, domain=[i, j])
    z_lin2[i, j] = z[i, j] <= y[i, t_pre]
    
    z_lin3 = gp.Equation(m, domain=[i, j])
    z_lin3[i, j] = z[i, j] <= y[j, t_post]
    
    z_lin4 = gp.Equation(m, domain=[i, j])
    z_lin4[i, j] = z[i, j] >= x[i, j] + y[i, t_pre] + y[j, t_post] - 2
    
    
    # Objective:
    total_emissions = (
        # 1. HQ -> FIRST RACE
        gp.Sum(Team, hq_emissions[Team, FIRST])
        
        # 2. Sum of all consecutive race emissions
        + gp.Sum([i, j], r_emissions[i, j] * x[i, j]) * 10
        
        # 3. Last race -> HQ
        + gp.Sum(Team, hq_emissions[Team, LAST])
    
        # ---- SUMMER BREAK ADJUSTMENTS ----
        
        # 4. REMOVE emission of break connection
        - gp.Sum([i, j], r_emissions[i, j] * z[i, j]) * 10
        
        # 5. ADD: pre-break race -> HQ
        + gp.Sum([Team, i], hq_emissions[Team, i] * y[i, t_pre])
        
        # 6. ADD: HQ -> post-break race
        + gp.Sum([Team, j], hq_emissions[Team, j] * y[j, t_post])
    
        # ---- LOOP CLOSURE ADJUSTMENT ----
        
        # 7. REMOVE LAST RACE -> FIRST RACE
        - r_emissions[LAST, FIRST] * x[LAST, FIRST] * 10
    )
    
    model = gp.Model(
        m,
        name="f1_calendar",
        equations=m.getEquations(),
        sense=gp.Sense.MIN,
        problem=gp.Problem.MIP,
        objective=total_emissions
    )
    
    display(model.solve(solver="gurobi", options=gp.Options(
        time_limit=time_limit,           # Default is 5 minutes
        relative_optimality_gap=0.05,
    )))

    result = {
                'first_race_circuit' : FIRST,
                'last_race_circuit': LAST,
                'objective_value': model.objective_value,
                'model_status': str(model.status),
                'solver_time': model.total_solver_time
            }
    
    print("-" * 70)

    if is_visualize:
        global x_baseline, u_baseline, y_baseline
        x_baseline = x.records
        u_baseline = u.records
        y_baseline = y.records
    
    return result

In [17]:
base_model_result = define_and_run_model(is_visualize=True)
base_model_result

,Solver Status,Model Status,Objective,Num of Equations,Num of Variables,Model Type,Solver,Solver Time
0,Normal,OptimalGlobal,5.116557e+06,4644,2242,MIP,GUROBI,108.583


----------------------------------------------------------------------


{'first_race_circuit': 'AUS',
 'last_race_circuit': 'ABD',
 'objective_value': 5116557.16273142,
 'model_status': 'ModelStatus.OptimalGlobal',
 'solver_time': 108.65699965506792}

## 3.2 Baseline Model Results & Visualization

### 3.1.1 Baseline Model Result

When the first race is fixed at Australia and last race at Abu Dhabi, the optimal minimum total CO2 emission is 5116557 kgCO2e.

### 3.2.2 Visualization:

We visualize the travel between race circuits starting from the first race using plotly.

In [18]:
## Install plotly package
%pip install plotly


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [19]:
import plotly.graph_objects as go

df = x_baseline                  # adjacency matrix in long form
df2 = u_baseline                 # each race with its order (u variable)
df1 = df[df['level'] == 1]      # edges actually used (x[i,j] = 1)

# Step 1: Race order
order_df = df2.sort_values("level").reset_index(drop=True)
order_df['rank'] = range(1, len(order_df) + 1)

# Step 2: Edges
edges = df1[['Circuit_0', 'Circuit_1']].reset_index(drop=True)

# Step 3: Coordinates
coords = circuits_df[['circuit_id', 'latitude', 'longitude']]

# Add rank to coordinates
coords = coords.merge(order_df[['Circuit', 'rank']], 
                      left_on='circuit_id', 
                      right_on='Circuit', 
                      how='left').drop(columns=['Circuit'])

# Create labels with rank
coords['label'] = coords['rank'].astype(int).astype(str) + '. ' + coords['circuit_id']

edges = (
    edges.merge(coords, left_on='Circuit_0', right_on='circuit_id')
         .rename(columns={'latitude':'lat_from', 'longitude':'lon_from'})
         .drop(columns=['circuit_id'])
         .merge(coords, left_on='Circuit_1', right_on='circuit_id')
         .rename(columns={'latitude':'lat_to', 'longitude':'lon_to'})
         .drop(columns=['circuit_id'])
)

fig = go.Figure()

# Draw route lines (without arrowheads)
for _, row in edges.iterrows():
    fig.add_trace(go.Scattergeo(
        lon=[row['lon_from'], row['lon_to']],
        lat=[row['lat_from'], row['lat_to']],
        mode='lines',
        line=dict(width=2, color='red'),
        opacity=0.8,
        name=f"{row['Circuit_0']} → {row['Circuit_1']}"
    ))

# Circuit nodes with rank labels
fig.add_trace(go.Scattergeo(
    lon=coords['longitude'],
    lat=coords['latitude'],
    mode='markers+text',
    marker=dict(size=8, color="blue"),
    text=coords['label'],
    textposition="top center",
    name="Circuits"
))

fig.update_layout(
    title="F1 Calendar Travel Route (Starting from AUS)",
    geo=dict(
        projection_type="natural earth",
        showcountries=True,
        landcolor="rgb(240, 240, 240)",
    ),
    height=650,
    showlegend=False,
    annotations=[
        dict(
            x=0.02,
            y=0.98,
            xref='paper',
            yref='paper',
            text='<b>Legend:</b><br>Number indicates visit order<br>(e.g., "1. AUS" = first race)',
            showarrow=False,
            bgcolor='white',
            bordercolor='black',
            borderwidth=1,
            borderpad=8,
            font=dict(size=11),
            align='left',
            xanchor='left',
            yanchor='top'
        )
    ]
)

fig.show()

## 3.3 F1 Race Calendar (baseline model)

From the below displayed race calendar table, we can infer the sequence of circuit travel. All teams travel from their HQ location to the first race (Albert Park Circuit, Australia) scheduled on 1st March 2026 and then travel to Marina Bay Street Circuit, Singapore for the 2nd race. From there on, they visit other circuits and have the final race at Yas Marina Circuit, Abu Dhabi on 6th December 2026.



In [20]:
df = y_baseline
df = df[df['level'] == 1][["Circuit", "Weekend"]]

final_weekends = climate_df[['week_num', 'race_date']]
final_weekends = final_weekends.copy()
final_weekends['week_num'] = final_weekends['week_num'].astype('category')


df['Weekend'] = df['Weekend'].astype(int)
final_weekends['week_num'] = final_weekends['week_num'].astype(int)
final_weekends_1 = pd.merge(df, final_weekends,
                            left_on='Weekend',
                            right_on='week_num',
                            how='inner')
final_weekends_1 = final_weekends_1.sort_values(by='Weekend')
final_weekends_1 = final_weekends_1[['Circuit', 'race_date']]
final_weekends_1
# Merge with circuits dataframe to include country, circuit_name, and city
final_weekends_1 = pd.merge(final_weekends_1, circuits_df,
                            left_on='Circuit',
                            right_on='circuit_id',
                            how='inner')
# Select the final columns you want
final_weekends_1 = final_weekends_1[['circuit_name', 'city', 'country',  'race_date']]
final_weekends_1

,circuit_name,city,country,race_date
0,Albert Park Circuit,Melbourne,Australia,2026-03-01
1,Marina Bay Street Circuit,Singapore,Singapore,2026-03-08
2,Suzuka International Racing Course,Suzuka,Japan,2026-03-22
3,Shanghai International Circuit,Shanghai,China,2026-04-05
4,Las Vegas Strip Circuit,Las Vegas,USA,2026-04-12
5,Autódromo Hermanos Rodríguez,Mexico City,Mexico,2026-05-17
6,Circuit of the Americas,Austin,USA,2026-05-24
7,Circuit Gilles Villeneuve,Montreal,Canada,2026-06-07
8,Miami International Autodrome,Miami,USA,2026-06-14
9,Autódromo José Carlos Pace,São Paulo,Brazil,2026-06-28


## 4 Senstivity Analysis

We conducted two targeted sensitivity analyses to evaluate how boundary conditions (start and end race locations) influence the overall optimal CO2 emission value.

### 4.1 Analysis 1 — Fixed Start (AUS), Variable End Location

In this scenario, the season opener is fixed to Australia (AUS), while the final race location is systematically varied across all candidate circuits. For each configuration, the optimization model is re-solved to observe how changes in the terminal race affect total travel emissions.


In [8]:
circuits = circuits_df["circuit_id"].tolist()

""" 
Senstivity Analysis Part 1:

Fix start race as AUS and vary last race location
"""

def run_analysis_1():
    """
    Analysis 1: Fix Start as AUS and run for all other circuits as ending
    """
    print("=" * 70)
    print("ANALYSIS 1: Fixed Start = AUS, Variable End")
    print("=" * 70)
    
    results_analysis_1 = []
    counter = 1
    for last_circuit in circuits:
        if last_circuit == 'AUS':  # Skip if start and end are the same
            continue
        
        print(f"\n Case {counter}: First race = AUS, Last race = {last_circuit} \n")
        try:
            result = define_and_run_model(FIRST='AUS', LAST=last_circuit)
            results_analysis_1.append(result)
        except Exception as e:
            print(f"  ERROR: {str(e)}")
            results_analysis_1.append({
                'first_circuit': 'AUS',
                'last_circuit': last_circuit,
                'objective_value': None,
                'solve_status': 'ERROR',
                'solver_time': None
            })
        finally:
            counter = counter + 1 
    
    df_analysis_1 = pd.DataFrame(results_analysis_1)
    return df_analysis_1

In [9]:
df_analysis_1 = run_analysis_1()
df_analysis_1

ANALYSIS 1: Fixed Start = AUS, Variable End

 Case 1: First race = AUS, Last race = BAH 



,Solver Status,Model Status,Objective,Num of Equations,Num of Variables,Model Type,Solver,Solver Time
0,Normal,OptimalGlobal,5.010846e+06,4644,2242,MIP,GUROBI,43.691


----------------------------------------------------------------------

 Case 2: First race = AUS, Last race = SAU 



,Solver Status,Model Status,Objective,Num of Equations,Num of Variables,Model Type,Solver,Solver Time
0,Normal,OptimalGlobal,4.910899e+06,4644,2242,MIP,GUROBI,129.567


----------------------------------------------------------------------

 Case 3: First race = AUS, Last race = JPN 



...

,Solver Status,Model Status,Objective,Num of Equations,Num of Variables,Model Type,Solver,Solver Time
0,Normal,OptimalGlobal,4.950483e+06,4644,2242,MIP,GUROBI,53.069


----------------------------------------------------------------------

 Case 23: First race = AUS, Last race = ABD 



,Solver Status,Model Status,Objective,Num of Equations,Num of Variables,Model Type,Solver,Solver Time
0,Normal,OptimalGlobal,5.116557e+06,4644,2242,MIP,GUROBI,114.018


----------------------------------------------------------------------


,first_race_circuit,last_race_circuit,objective_value,model_status,solver_time
0,AUS,BAH,5.010846e+06,ModelStatus.OptimalGlobal,43.793000
1,AUS,SAU,4.910899e+06,ModelStatus.OptimalGlobal,129.622000
2,AUS,JPN,5.297792e+06,ModelStatus.OptimalGlobal,136.143000
3,AUS,CHN,5.364361e+06,ModelStatus.OptimalGlobal,286.393000
4,AUS,MIA,5.236700e+06,ModelStatus.OptimalGlobal,214.571000
5,AUS,MON,4.849630e+06,ModelStatus.OptimalGlobal,258.666000
6,AUS,CAN,1.500134e+06,ModelStatus.InfeasibleGlobal,6.247000
7,AUS,ESP,5.050128e+06,ModelStatus.Feasible,300.107000
8,AUS,MAD,1.141816e+06,ModelStatus.InfeasibleGlobal,7.083000
9,AUS,AUT,1.138785e+06,ModelStatus.InfeasibleGlobal,5.235000


In [10]:
def print_senstivity_analysis(df_analysis_result, is_start_fixed=True):
        df_analysis_result = df_analysis_result[df_analysis_result['model_status'] == "ModelStatus.OptimalGlobal"].sort_values("objective_value")
        min_idx = df_analysis_result['objective_value'].idxmin()
        best_row = df_analysis_result.loc[min_idx]
        if is_start_fixed:
                print(f"When the start race circuit is fixed as {best_row['first_race_circuit']}, the minimal CO2 emission occurs when last race circuit is {best_row['last_race_circuit']}, \nand has a total emission value of {best_row['objective_value']} kg CO₂e")
        else:
                print(f"When the end race circuit is fixed as {best_row['last_race_circuit']} and considering only normal solver status, the minimal CO2 emission occurs when first race circuit is {best_row['first_race_circuit']}, \nand has a total emission value of {best_row['objective_value']} kg CO₂e")

print_senstivity_analysis(df_analysis_1)
df_analysis_1.to_csv("senstivity_analysis_1.csv")

When the start race circuit is fixed as AUS, the minimal CO2 emission occurs when last race circuit is MON, 
and has a total emission value of 4849630.0952396 kg CO₂e


### 4.2 Analysis 2 — Fixed End (ABD), Variable Start Location

Here, the final race is fixed to Abu Dhabi (ABD), and the starting location is varied. This assesses how the choice of season opener shapes early-season travel patterns, cumulative emissions, and downstream routing decisions. By comparing objective values across alternative starting points, we quantify the sensitivity of the model to the first race placement.

In [11]:
"""
Analysis 2: Fix End as ABD and vary the start race location.
"""
def run_analysis_2():
    print("\n" + "=" * 70)
    print("ANALYSIS 2: Variable Start, Fixed End = ABD")
    print("=" * 70)
    
    results_analysis_2 = []
    counter = 1
    for first_circuit in circuits:
        if first_circuit == 'ABD':  # Skip if start and end are the same
            continue

        print(f"\n Case {counter}: First race = {first_circuit}, Last race = ABD \n")
        try:
            result = define_and_run_model(FIRST=first_circuit, LAST='ABD')
            results_analysis_2.append(result)
        except Exception as e:
            print(f"  ERROR: {str(e)}")
            results_analysis_2.append({
                'first_circuit': first_circuit,
                'last_circuit': 'ABD',
                'objective_value': None,
                'solve_status': 'ERROR',
                'solver_time': None
            })
        finally:
            counter = counter + 1 
    
    df_analysis_2 = pd.DataFrame(results_analysis_2)
    return df_analysis_2

In [12]:
df_analysis_2 = run_analysis_2()
df_analysis_2


ANALYSIS 2: Variable Start, Fixed End = ABD

 Case 1: First race = BAH, Last race = ABD 



,Solver Status,Model Status,Objective,Num of Equations,Num of Variables,Model Type,Solver,Solver Time
0,Normal,OptimalGlobal,4.735397e+06,4644,2242,MIP,GUROBI,75.168


----------------------------------------------------------------------

 Case 2: First race = SAU, Last race = ABD 



,Solver Status,Model Status,Objective,Num of Equations,Num of Variables,Model Type,Solver,Solver Time
0,Normal,OptimalGlobal,4.691860e+06,4644,2242,MIP,GUROBI,197.129


----------------------------------------------------------------------

 Case 3: First race = AUS, Last race = ABD 



...

----------------------------------------------------------------------

 Case 23: First race = QAT, Last race = ABD 



[MODEL - WARNING] The solve was interrupted! Solve status: ResourceInterrupt. For further information, see https://gamspy.readthedocs.io/en/latest/reference/gamspy._model.html#gamspy.SolveStatus.


,Solver Status,Model Status,Objective,Num of Equations,Num of Variables,Model Type,Solver,Solver Time
0,Resource,Feasible,5.092346e+06,4644,2242,MIP,GUROBI,300.07


----------------------------------------------------------------------


,first_race_circuit,last_race_circuit,objective_value,model_status,solver_time
0,BAH,ABD,4.735397e+06,ModelStatus.OptimalGlobal,75.218000
1,SAU,ABD,4.691860e+06,ModelStatus.OptimalGlobal,197.184000
2,AUS,ABD,5.116557e+06,ModelStatus.OptimalGlobal,105.448000
3,JPN,ABD,4.675517e+06,ModelStatus.OptimalGlobal,53.742000
4,CHN,ABD,4.708203e+06,ModelStatus.OptimalGlobal,170.124000
5,MIA,ABD,4.691969e+06,ModelStatus.Feasible,300.149000
6,MON,ABD,4.527364e+06,ModelStatus.Feasible,300.098000
7,CAN,ABD,7.243858e+05,ModelStatus.InfeasibleGlobal,6.473000
8,ESP,ABD,4.497991e+06,ModelStatus.Feasible,300.153000
9,MAD,ABD,4.827470e+06,ModelStatus.Feasible,300.098000


In [13]:

print_senstivity_analysis(df_analysis_2, False)
df_analysis_2.to_csv("senstivity_analysis_2.csv")

When the end race circuit is fixed as ABD and considering only normal solver status, the minimal CO2 emission occurs when first race circuit is AZE, 
and has a total emission value of 4657162.231291912 kg CO₂e


# 5. Conclusion

This project successfully developed and solved a mixed-integer optimization model to design a lower-emission F1 race calendar by minimizing cumulative CO₂e from inter-race travel while respecting strict seasonal, geographic, and operational constraints. The baseline optimization produced an optimal annual travel footprint of 5,116,557 kgCO₂e, yielding a season that begins at Albert Park Circuit (AUS) on 1 March 2026 and concludes at Yas Marina Circuit (ABD) on 6 December 2026. The resulting schedule demonstrates that enforcing geographically coherent race sequences significantly reduces unnecessary long-haul transitions, confirming that structured routing is a powerful lever for sustainability in global motorsport logistics.

The sensitivity analysis further highlights that the model is highly dependent on start and end circuit selection, emphasizing their influence on the overall trajectory and resulting emissions. When fixing  Albert Park Circuit, Australia (AUS) as the starting race, the lowest total emissions occurred when Circuit de Monaco (MON) was set as the final race, producing 4,849,630 kgCO₂e. Conversely, when fixing Yas Marina Circuit, Abu Dhabi (ABD) as the final circuit and varying the starting point, the optimal configuration began at Baku City Circuit (AZE), reducing total emissions even further to 4,657,162.23 kgCO₂e. These results demonstrate that early-season positioning imposes a stronger structural constraint on the calendar’s geographic progression than the final race, and that strategically selecting start/end points can reduce emissions by 5–8.9% relative to the baseline. For the F1 calendar schedule, kindly refer to Section 3.3.


Overall, the project demonstrates that optimization-driven scheduling offers a robust framework for designing environmentally efficient racing calendars. The findings reinforce the critical role of boundary conditions (starting and ending circuits) in shaping season-long travel patterns, and they offer a data-driven foundation for scheduling the F1 race season.


# 6. Future Work

A promising direction for future enhancement is to incorporate heterogeneous freight profiles by allowing the transported weight per team to vary rather than assuming a uniform 50-ton load. Introducing team-specific logistics parameters would improve the granularity of emission estimates.